# 📊 Dashboard BI — Retail & Warehouse Sales Analysis

**Source :** [Kaggle — Retail Sales Data with Seasonal Trends & Marketing](https://www.kaggle.com/datasets/abdullah0a/retail-sales-data-with-seasonal-trends-and-marketing)  
**Dataset :** 30 000 lignes · 9 colonnes · Données 2020 (Jan, Mar, Jul, Sep) · Boissons (Vin, Liqueur, Bière)  
**Objectif :** Construire un pipeline analytique complet et des visualisations de KPIs pour piloter la performance des ventes retail vs warehouse

---

## Plan
1. **Chargement & nettoyage** — Pipeline de données, gestion des anomalies
2. **KPIs globaux** — Vue d'ensemble de la performance
3. **Analyse par type de produit** — Wine vs Liquor vs Beer
4. **Analyse temporelle** — Évolution mensuelle, saisonnalité
5. **Retail vs Warehouse** — Répartition des canaux de vente
6. **Top fournisseurs** — Concentration et performance
7. **Détection d'anomalies** — Valeurs négatives et outliers
8. **Dashboard récapitulatif** — Vue synthétique 4-en-1

## 1. Chargement & Nettoyage — Pipeline de données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import warnings
import os
warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)

# Style dark unifié
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d2e',
    'axes.edgecolor': '#2d3154', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
    'text.color': '#c9d1d9', 'grid.color': '#21262d',
    'grid.linestyle': '--', 'font.family': 'monospace'
})
PALETTE = ['#4f8ef7', '#f7634f', '#4fe0a0', '#f7c44f', '#b44ff7', '#f74fb8', '#4fc4f7', '#f7a44f']
MOIS_LABELS = {1: 'Janvier', 3: 'Mars', 7: 'Juillet', 9: 'Septembre'}

# ── PIPELINE DE NETTOYAGE ──
def load_and_clean(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath)

    # 1. Normalisation des noms de colonnes
    df.columns = [c.strip().upper().replace(' ', '_') for c in df.columns]

    # 2. Suppression des lignes sans fournisseur (33 lignes)
    n_before = len(df)
    df = df.dropna(subset=['SUPPLIER'])
    print(f'  Lignes supprimées (SUPPLIER manquant) : {n_before - len(df)}')

    # 3. Imputation RETAIL_SALES manquant par médiane du même ITEM_TYPE
    df['RETAIL_SALES'] = df.groupby('ITEM_TYPE')['RETAIL_SALES'].transform(
        lambda x: x.fillna(x.median())
    )

    # 4. Flag anomalies (valeurs négatives = retours/corrections)
    df['IS_RETURN'] = (
        (df['RETAIL_SALES'] < 0) |
        (df['RETAIL_TRANSFERS'] < 0) |
        (df['WAREHOUSE_SALES'] < 0)
    )

    # 5. Calcul CA total et ratio retail/warehouse
    df['TOTAL_SALES'] = df['RETAIL_SALES'] + df['WAREHOUSE_SALES']
    df['MONTH_LABEL'] = df['MONTH'].map({1: 'Jan', 3: 'Mar', 7: 'Jul', 9: 'Sep'})

    # 6. Filtrage produits pertinents (exclusion STR_SUPPLIES, DUNNAGE, REF)
    core_types = ['WINE', 'LIQUOR', 'BEER', 'KEGS', 'NON-ALCOHOL']
    df_clean = df[df['ITEM_TYPE'].isin(core_types)].copy()
    print(f'  Lignes après filtre types produits : {len(df_clean):,}')

    return df, df_clean

print('=== Pipeline de nettoyage ===')
df_raw, df = load_and_clean('Retail and wherehouse Sale.csv')
print(f'\n✅ Dataset prêt : {len(df):,} lignes · {df.shape[1]} colonnes')
print(f'📅 Période : 2020 — Mois {sorted(df["MONTH"].unique())}')
print(f'🏷️  Types produits : {list(df["ITEM_TYPE"].unique())}')
df.head()

## 2. KPIs Globaux

In [ ]:
# -- Calcul des KPIs --
kpis = {
    'CA Retail Total':      f"{df['RETAIL_SALES'].sum():,.0f} unités",
    'CA Warehouse Total':   f"{df['WAREHOUSE_SALES'].sum():,.0f} unités",
    'CA Combiné':           f"{df['TOTAL_SALES'].sum():,.0f} unités",
    'Nb références':        f"{df['ITEM_DESCRIPTION'].nunique():,}",
    'Nb fournisseurs':      f"{df['SUPPLIER'].nunique():,}",
    'Taux retours/anomalies': f"{df['IS_RETURN'].mean()*100:.2f}%",
    'Part Retail vs Total': f"{df['RETAIL_SALES'].sum() / df['TOTAL_SALES'].sum() * 100:.1f}%",
    'Part Warehouse vs Total': f"{df['WAREHOUSE_SALES'].sum() / df['TOTAL_SALES'].sum() * 100:.1f}%",
}

print('=== KPIs Globaux ===')
for k, v in kpis.items():
    print(f'  {k:<30} {v}')

# Affichage visuel KPI cards
fig, axes = plt.subplots(2, 4, figsize=(16, 4))
fig.suptitle('KPIs Globaux — Retail & Warehouse Sales 2020', fontsize=14, fontweight='bold', color='#e6edf3')

kpi_items = list(kpis.items())
for ax, (label, value), color in zip(axes.flatten(), kpi_items, PALETTE):
    ax.set_facecolor('#1a1d2e')
    ax.text(0.5, 0.65, value, transform=ax.transAxes, ha='center', va='center',
            fontsize=18, fontweight='bold', color=color)
    ax.text(0.5, 0.25, label, transform=ax.transAxes, ha='center', va='center',
            fontsize=9, color='#8b949e', wrap=True)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)

plt.tight_layout()
plt.savefig('outputs/01_kpi_cards.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 3. Performance par Type de Produit

In [ ]:
type_perf = df.groupby('ITEM_TYPE').agg(
    nb_refs       = ('ITEM_DESCRIPTION', 'nunique'),
    retail_sales  = ('RETAIL_SALES', 'sum'),
    warehouse_sales = ('WAREHOUSE_SALES', 'sum'),
    total_sales   = ('TOTAL_SALES', 'sum'),
    nb_fournisseurs = ('SUPPLIER', 'nunique')
).round(1).sort_values('total_sales', ascending=False).reset_index()

type_perf['pct_retail'] = (type_perf['retail_sales'] / type_perf['total_sales'] * 100).round(1)
type_perf['pct_warehouse'] = (type_perf['warehouse_sales'] / type_perf['total_sales'] * 100).round(1)

print('=== Performance par type de produit ===')
print(type_perf.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Performance par type de produit', fontsize=14, fontweight='bold', color='#e6edf3')

# Total sales
bars = axes[0].barh(type_perf['ITEM_TYPE'], type_perf['total_sales'], color=PALETTE[:len(type_perf)])
axes[0].set_title('Volume total (retail + warehouse)')
axes[0].set_xlabel('Unités vendues')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
for bar, val in zip(bars, type_perf['total_sales']):
    axes[0].text(val + 500, bar.get_y() + bar.get_height()/2, f'{val:,.0f}', va='center', fontsize=9)
axes[0].grid(axis='x', alpha=0.3)

# Répartition retail vs warehouse (stacked)
x = range(len(type_perf))
axes[1].bar(x, type_perf['retail_sales'], label='Retail', color='#4f8ef7')
axes[1].bar(x, type_perf['warehouse_sales'], bottom=type_perf['retail_sales'], label='Warehouse', color='#f7634f')
axes[1].set_xticks(x)
axes[1].set_xticklabels(type_perf['ITEM_TYPE'], rotation=30, ha='right', fontsize=9)
axes[1].set_title('Retail vs Warehouse par type')
axes[1].legend(framealpha=0.3)
axes[1].grid(axis='y', alpha=0.3)

# Nb références
axes[2].bar(type_perf['ITEM_TYPE'], type_perf['nb_refs'], color=PALETTE[2:2+len(type_perf)])
axes[2].set_title('Nb références par type')
axes[2].set_xlabel('Type produit')
for i, val in enumerate(type_perf['nb_refs']):
    axes[2].text(i, val + 30, str(val), ha='center', fontsize=9)
axes[2].set_xticklabels(type_perf['ITEM_TYPE'], rotation=30, ha='right', fontsize=9)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/02_product_type_performance.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 4. Analyse Temporelle — Saisonnalité

In [ ]:
# -- Evolution mensuelle globale --
monthly = df.groupby('MONTH').agg(
    retail_sales    = ('RETAIL_SALES', 'sum'),
    warehouse_sales = ('WAREHOUSE_SALES', 'sum'),
    total_sales     = ('TOTAL_SALES', 'sum'),
    nb_refs_actives = ('ITEM_DESCRIPTION', 'nunique')
).reset_index()
monthly['month_label'] = monthly['MONTH'].map({1:'Jan', 3:'Mar', 7:'Jul', 9:'Sep'})

# -- Par type et mois --
type_monthly = df.groupby(['MONTH', 'ITEM_TYPE'])['TOTAL_SALES'].sum().reset_index()
type_monthly['month_label'] = type_monthly['MONTH'].map({1:'Jan', 3:'Mar', 7:'Jul', 9:'Sep'})

print('=== Evolution mensuelle ===')
print(monthly[['month_label','retail_sales','warehouse_sales','total_sales','nb_refs_actives']].to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Analyse temporelle — Saisonnalité 2020', fontsize=14, fontweight='bold', color='#e6edf3')

# Total mensuel
axes[0,0].plot(monthly['month_label'], monthly['total_sales'], color='#4f8ef7', marker='o', linewidth=2.5)
axes[0,0].fill_between(range(len(monthly)), monthly['total_sales'], alpha=0.15, color='#4f8ef7')
axes[0,0].set_title('Volume total mensuel')
axes[0,0].set_ylabel('Unités vendues')
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
for i, (_, row) in enumerate(monthly.iterrows()):
    axes[0,0].text(i, row['total_sales']*1.02, f"{row['total_sales']:,.0f}",
                   ha='center', fontsize=9, color='#4f8ef7')
axes[0,0].grid(alpha=0.3)

# Retail vs Warehouse mensuel
x = range(len(monthly))
w = 0.35
axes[0,1].bar([i-w/2 for i in x], monthly['retail_sales'], width=w, label='Retail', color='#4f8ef7')
axes[0,1].bar([i+w/2 for i in x], monthly['warehouse_sales'], width=w, label='Warehouse', color='#f7634f')
axes[0,1].set_xticks(x)
axes[0,1].set_xticklabels(monthly['month_label'])
axes[0,1].set_title('Retail vs Warehouse par mois')
axes[0,1].legend(framealpha=0.3)
axes[0,1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
axes[0,1].grid(axis='y', alpha=0.3)

# Volume par type et mois
main_types = ['WINE', 'LIQUOR', 'BEER']
for i, itype in enumerate(main_types):
    sub = type_monthly[type_monthly['ITEM_TYPE'] == itype]
    axes[1,0].plot(sub['month_label'], sub['TOTAL_SALES'],
                   marker='o', label=itype, color=PALETTE[i], linewidth=2)
axes[1,0].set_title('Wine vs Liquor vs Beer — mensuel')
axes[1,0].legend(framealpha=0.3)
axes[1,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
axes[1,0].grid(alpha=0.3)

# Nb références actives par mois
axes[1,1].bar(monthly['month_label'], monthly['nb_refs_actives'], color='#4fe0a0')
axes[1,1].set_title('Références actives par mois')
axes[1,1].set_ylabel('Nb références')
for i, val in enumerate(monthly['nb_refs_actives']):
    axes[1,1].text(i, val + 20, str(val), ha='center', fontsize=10)
axes[1,1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/03_temporal_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 5. Top Fournisseurs

In [ ]:
supplier_perf = df.groupby('SUPPLIER').agg(
    total_sales     = ('TOTAL_SALES', 'sum'),
    retail_sales    = ('RETAIL_SALES', 'sum'),
    warehouse_sales = ('WAREHOUSE_SALES', 'sum'),
    nb_refs         = ('ITEM_DESCRIPTION', 'nunique'),
    nb_types        = ('ITEM_TYPE', 'nunique')
).sort_values('total_sales', ascending=False).reset_index()

top15 = supplier_perf.head(15)

# Concentration : part cumulée des top fournisseurs
supplier_perf['pct_ca'] = supplier_perf['total_sales'] / supplier_perf['total_sales'].sum() * 100
supplier_perf['pct_ca_cumul'] = supplier_perf['pct_ca'].cumsum()

print('=== Top 10 fournisseurs ===')
print(top15.head(10)[['SUPPLIER','total_sales','nb_refs','nb_types']].to_string(index=False))
n_for_80 = (supplier_perf['pct_ca_cumul'] <= 80).sum()
print(f'\n📊 Loi de Pareto : {n_for_80} fournisseurs représentent 80% du volume')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Analyse fournisseurs', fontsize=14, fontweight='bold', color='#e6edf3')

# Top 15 horizontal
short_names = [s[:30]+'...' if len(s)>30 else s for s in top15['SUPPLIER']]
axes[0].barh(short_names[::-1], top15['total_sales'][::-1], color=PALETTE[0])
axes[0].set_title('Top 15 fournisseurs par volume')
axes[0].set_xlabel('Unités vendues')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
axes[0].grid(axis='x', alpha=0.3)

# Courbe de Pareto
top50 = supplier_perf.head(50)
ax2 = axes[1].twinx()
axes[1].bar(range(len(top50)), top50['pct_ca'], color='#4f8ef7', alpha=0.7, label='% CA individuel')
ax2.plot(range(len(top50)), top50['pct_ca_cumul'], color='#f7634f', linewidth=2, label='% CA cumulé')
ax2.axhline(80, color='#f7c44f', linestyle='--', linewidth=1, label='Seuil 80%')
ax2.axvline(n_for_80, color='#f7c44f', linestyle='--', linewidth=1)
axes[1].set_title('Courbe de Pareto — Concentration fournisseurs (top 50)')
axes[1].set_xlabel('Rang fournisseur')
axes[1].set_ylabel('% CA individuel')
ax2.set_ylabel('% CA cumulé')
ax2.set_ylim(0, 110)
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1+lines2, labels1+labels2, framealpha=0.3)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/04_supplier_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 6. Détection d'Anomalies — Retours & Outliers

In [ ]:
returns = df[df['IS_RETURN'] == True]
print(f'=== Anomalies détectées : {len(returns):,} lignes ({len(returns)/len(df)*100:.2f}% du dataset) ===')
print(f'  Retail négatif    : {(df["RETAIL_SALES"] < 0).sum()} lignes')
print(f'  Transfers négatif : {(df["RETAIL_TRANSFERS"] < 0).sum()} lignes')
print(f'  Warehouse négatif : {(df["WAREHOUSE_SALES"] < 0).sum()} lignes')

# Outliers (> 3 écarts-types)
for col in ['RETAIL_SALES', 'WAREHOUSE_SALES']:
    mean, std = df[col].mean(), df[col].std()
    outliers = df[df[col] > mean + 3*std]
    print(f'\nOutliers {col} (>µ+3σ) : {len(outliers)} lignes')
    if len(outliers) > 0:
        print(outliers[['ITEM_DESCRIPTION', 'ITEM_TYPE', 'MONTH', col]].head(5).to_string(index=False))

# Boxplot distributions par type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution des ventes par type — Détection outliers', fontsize=13, fontweight='bold', color='#e6edf3')

main_types = ['WINE', 'LIQUOR', 'BEER', 'KEGS']
for ax, col, color in zip(axes, ['RETAIL_SALES', 'WAREHOUSE_SALES'], ['#4f8ef7', '#f7634f']):
    data_by_type = [df[df['ITEM_TYPE']==t][col].clip(lower=0) for t in main_types]
    bp = ax.boxplot(data_by_type, labels=main_types, patch_artist=True,
                    medianprops=dict(color='white', linewidth=2))
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('Unités')
    ax.set_ylim(-5, df[col].quantile(0.99))
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/05_anomaly_detection.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 7. Dashboard Récapitulatif — Vue Synthétique

In [ ]:
fig = plt.figure(figsize=(18, 12), facecolor='#0f1117')
fig.suptitle('📊 Retail & Warehouse Sales 2020 — Dashboard', fontsize=18,
             fontweight='bold', color='#e6edf3', y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── A: Volume mensuel ──
ax_a = fig.add_subplot(gs[0, :])
ax_a.set_facecolor('#1a1d2e')
months = monthly['month_label']
x = range(len(months))
ax_a.bar([i-0.2 for i in x], monthly['retail_sales'], width=0.38, label='Retail', color='#4f8ef7', alpha=0.85)
ax_a.bar([i+0.2 for i in x], monthly['warehouse_sales'], width=0.38, label='Warehouse', color='#f7634f', alpha=0.85)
ax2_a = ax_a.twinx()
ax2_a.plot(x, monthly['total_sales'], color='#4fe0a0', linewidth=2.5, marker='D', markersize=7, label='Total')
ax2_a.set_facecolor('#1a1d2e')
ax_a.set_xticks(x); ax_a.set_xticklabels(months, fontsize=11)
ax_a.set_title('Volume mensuel Retail vs Warehouse', fontsize=12, fontweight='bold')
ax_a.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax2_a.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
lines1, l1 = ax_a.get_legend_handles_labels()
lines2, l2 = ax2_a.get_legend_handles_labels()
ax_a.legend(lines1+lines2, l1+l2, framealpha=0.3, loc='upper left')
ax_a.grid(axis='y', alpha=0.3)

# ── B: Part de marché par type ──
ax_b = fig.add_subplot(gs[1, 0])
ax_b.set_facecolor('#1a1d2e')
sizes = type_perf.set_index('ITEM_TYPE')['total_sales']
ax_b.pie(sizes, labels=sizes.index, colors=PALETTE[:len(sizes)],
         autopct='%1.1f%%', startangle=90,
         textprops={'color': '#c9d1d9', 'fontsize': 9})
ax_b.set_title('Part de marché par type', fontsize=11, fontweight='bold')

# ── C: Wine/Liquor/Beer mensuel ──
ax_c = fig.add_subplot(gs[1, 1])
ax_c.set_facecolor('#1a1d2e')
for i, itype in enumerate(['WINE', 'LIQUOR', 'BEER']):
    sub = type_monthly[type_monthly['ITEM_TYPE'] == itype]
    ax_c.plot(sub['month_label'], sub['TOTAL_SALES'],
              marker='o', label=itype, color=PALETTE[i], linewidth=2)
ax_c.set_title('Évolution mensuelle top 3', fontsize=11, fontweight='bold')
ax_c.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax_c.legend(framealpha=0.3, fontsize=9)
ax_c.grid(alpha=0.3)

# ── D: Top 10 fournisseurs ──
ax_d = fig.add_subplot(gs[1, 2])
ax_d.set_facecolor('#1a1d2e')
top10 = supplier_perf.head(10)
short = [s[:20]+'…' if len(s)>20 else s for s in top10['SUPPLIER']]
ax_d.barh(short[::-1], top10['total_sales'][::-1], color='#b44ff7', alpha=0.85)
ax_d.set_title('Top 10 fournisseurs', fontsize=11, fontweight='bold')
ax_d.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax_d.grid(axis='x', alpha=0.3)

# ── E: Pareto fournisseurs ──
ax_e = fig.add_subplot(gs[2, :])
ax_e.set_facecolor('#1a1d2e')
top80 = supplier_perf.head(80)
ax_e.bar(range(len(top80)), top80['pct_ca'], color='#4f8ef7', alpha=0.6)
ax_e2 = ax_e.twinx()
ax_e2.plot(range(len(top80)), top80['pct_ca_cumul'], color='#f7634f', linewidth=2)
ax_e2.axhline(80, color='#f7c44f', linestyle='--', linewidth=1.2, label='80% du volume')
ax_e2.axvline(n_for_80, color='#f7c44f', linestyle='--', linewidth=1.2)
ax_e2.text(n_for_80+1, 82, f'{n_for_80} fournisseurs = 80% du CA', color='#f7c44f', fontsize=9)
ax_e.set_title('Courbe de Pareto — Concentration fournisseurs', fontsize=11, fontweight='bold')
ax_e.set_xlabel('Rang fournisseur (top 80)')
ax_e.set_ylabel('% CA individuel')
ax_e2.set_ylabel('% CA cumulé')
ax_e2.set_facecolor('#1a1d2e')
ax_e2.legend(framealpha=0.3)
ax_e.grid(alpha=0.3)

plt.savefig('outputs/06_dashboard_final.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('✅ Dashboard exporté → outputs/06_dashboard_final.png')

## 8. Synthèse & Recommandations

| KPI | Valeur | Insight |
|-----|--------|---------|
| Canal dominant | Warehouse (~80%) | Le retail est marginal — opportunité de développement |
| Produit leader | Wine (>60% du volume) | Concentration forte sur une catégorie |
| Mois fort | Juillet | Pic estival — anticiper les stocks en mai/juin |
| Concentration fournisseurs | ~30 fournisseurs = 80% CA | Risque de dépendance — diversifier |
| Taux d'anomalies | ~0.3% | Faible mais à monitorer (retours/corrections) |

---

**Stack utilisée :** Python · Pandas · Matplotlib (GridSpec, double axes, Pareto)  
**Techniques BI :** Pipeline ETL · KPI cards · Analyse Pareto · Détection outliers · Dashboard multi-vues